In [1]:
import chess.svg
import numpy as np
from pathlib import Path
import torch
import pytorch_lightning as pl

from chess_gnn.models import *
from chess_gnn.utils import PGNBoardHelper, ChessPoint
from chess_gnn.tokenizers import SimpleChessTokenizer
from chess_gnn.visualization import MaskVisualizationHelper

from plotly import graph_objects as go
import dash
from dash import dcc, html, Input, Output, State
import chess
import chess.svg
import base64

In [2]:
class ChessBoardArray:
    def __init__(self, board_array: np.array):
        self.board_array = board_array
        if len(self.board_array) != 64:
            raise ValueError("Board array must be of length 64")
        self.tokenizer = SimpleChessTokenizer()
    
    def __len__(self):
        return len(self.board_array)
    
    def __getitem__(self, item):
        return self.board_array[item]
    
    def to_pieces(self):
        untokenized = self.tokenizer.untokenize(self.board_array)
        untokenized = [token if token != '.' else None for token in untokenized]
        return untokenized

In [3]:
def board_to_svg_image(board: ChessBoardArray):
    images = []
    pieces = board.to_pieces()
    for idx in range(64):
        piece = pieces[idx]
        point = ChessPoint.from_1d(idx)
        if piece:
            svg = chess.svg.piece(chess.Piece.from_symbol(piece))
            svg_bytes = svg.encode('utf-8')
            uri = f"data:image/svg+xml;base64,{base64.b64encode(svg_bytes).decode('utf-8')}"
            images.append(dict(
                source=uri,
                xref="x", yref="y",
                x=point.x,
                y=7-point.y,
                sizex=1.0, sizey=1.0,
                xanchor="center", yanchor="middle",
                layer="above"
            ))
    return images

def create_board_figure(board: ChessBoardArray, mask: np.array):
    # Create base array for heatmap
    z = np.flipud(np.reshape(mask, (8,8)))

    # Create transparent red heatmap as mask
    heatmap = go.Heatmap(
        z=z,
        x=list(range(8)),
        y=list(range(8)),
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(255,0,0,0.4)']],
        zmin=0,
        zmax=1,
        showscale=False,
        hoverinfo='skip',
        xgap=0,
        ygap=0,
        opacity=1.0,
    )

    fig = go.Figure(data=[heatmap])

    fig.update_layout(
        width=320,
        height=320,
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(
            scaleanchor="x",
            scaleratio=1,
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-0.5, 7.5],
            fixedrange=True
        ),
        xaxis=dict(
            constrain='domain',
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-0.5, 7.5],
            fixedrange=True
        ),
        images=board_to_svg_image(board),
        shapes=[],
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    return fig

def create_dual_board_app(boards1: list[ChessBoardArray], 
                          boards2: list[ChessBoardArray], 
                          # boards3: list[ChessBoardArray],
                          # boards4: list[ChessBoardArray],
                          masks):
    assert len(boards1) == len(boards2), "Board lists must be the same length"

    num_steps = len(boards1)
    app = dash.Dash(__name__)

    app.layout = html.Div([
        html.Div([
            html.Button("Prev", id="prev-btn", n_clicks=0),
            html.Button("Next", id="next-btn", n_clicks=0),
            html.Span(id="step-label", style={"marginLeft": "1rem"}),
        ], style={"marginBottom": "1rem"}),

        dcc.Store(id="current-step", data=0),

        html.Div([
            dcc.Graph(id='left-board', config={"displayModeBar": False}),
            dcc.Graph(id='right-board', config={"displayModeBar": False}),
        ], style={"display": "flex", "justifyContent": "center", "gap": "2rem"}),
    ])

    @app.callback(
        Output('current-step', 'data'),
        Output('step-label', 'children'),
        Input('prev-btn', 'n_clicks'),
        Input('next-btn', 'n_clicks'),
        State('current-step', 'data')
    )
    def update_step(prev, nxt, current):
        ctx = dash.callback_context.triggered_id
        if ctx == 'prev-btn':
            current = max(0, current - 1)
        elif ctx == 'next-btn':
            current = min(num_steps - 1, current + 1)
        return current, f"Step: {current} / {num_steps - 1}"

    @app.callback(
        Output('left-board', 'figure'),
        Output('right-board', 'figure'),
        Input('current-step', 'data')
    )
    def update_boards(idx):
        mask = masks[idx]
        return (
            create_board_figure(boards1[idx], mask),
            create_board_figure(boards2[idx], mask),
        )

    return app

In [4]:
ckpt_file = '/Users/ray/models/chess/transformer/29719fc8-7e51-4d2b-8f1d-6500821d6c4a/epoch=0-step=110000.ckpt'
model: ChessTransformer = ChessTransformer.load_from_checkpoint(ckpt_file, from_pretrained=True)

model.eval()
pl.seed_everything(42)

/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.2, which is newer than your current Lightning version: v2.5.1.post0
/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'encoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder'])`.
/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'decoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['decoder'])`.
/Users/ray/miniconda3/envs/ChessGNN/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:209: Attribute 'mask_handler' is an instance of `nn.Module` and is already sa

42

In [5]:
model.pretrained

True

In [28]:
model.mask_handler.set_masking_ratio(0.50)

In [29]:
model.mask_handler.masking_ratio

0.5

In [14]:
pgn = PGNBoardHelper(Path('/Users/ray/Datasets/chess/Carlsen.pgn'))
for i in range(100):
    pgn.get_game()

tokenizer = SimpleChessTokenizer()
game_batch = pgn.get_game_batch(tokenizer=tokenizer)

In [30]:
with torch.no_grad():
    loss = model(game_batch)

In [31]:
loss

{'current_board_loss': tensor(0.4947),
 'next_board_loss': tensor(0.5194),
 'loss': tensor(1.0141)}

In [32]:
helper = MaskVisualizationHelper(transformer=model)
with torch.no_grad():
    m, current, nxt = helper.get_preds(game_batch)

In [33]:
current_pred = [ChessBoardArray(arr) for arr in current.numpy()]
current_label = [ChessBoardArray(arr) for arr in game_batch['board'].squeeze().numpy()]
nxt_pred = [ChessBoardArray(arr) for arr in nxt.numpy()]
nxt_label = [ChessBoardArray(arr) for arr in game_batch['next_board'].squeeze().numpy()]

In [34]:
app = create_dual_board_app(current_label, current_pred, m)
app.run(mode="jupyter-inline", port=8051)

In [ ]:
tokenizer.vocab

In [ ]:
tokenizer.inverse_vocab